# Module 10 - 실습 2: Annotated Reducer 패턴

## 학습 목표
- LangGraph의 `Annotated` + Reducer 패턴을 이해한다
- `operator.add`로 리스트를 누적하는 패턴을 구현할 수 있다
- 노드가 변경된 필드만 반환하는 효율적인 패턴을 적용할 수 있다

## 참조 자료
- 학습 문서: `docs/part4-production/10-resource-optimization.md` (섹션 2.3)

## 개념 요약

**문제**: 매 노드마다 전체 State를 복사하면 메모리 낭비
```python
# Before (비효율적)
def my_node(state):
    return {**state, "result": compute()}  # 전체 복사!
```

**해결**: Annotated Reducer로 변경된 필드만 반환
```python
# After (효율적)
class MyState(TypedDict):
    result: Annotated[str | None, _replace]  # reducer 지정
    messages: Annotated[list, operator.add]  # 리스트 누적

def my_node(state):
    return {"result": compute()}  # 변경 필드만 반환!
```

In [ ]:
import sys
sys.path.insert(0, '..')

import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, END
from common.fake_llm import FakeLLM

print("Annotated Reducer 실습 준비 완료")

## 실습 1: _replace reducer 정의

기존 값을 새 값으로 교체하는 `_replace` reducer 함수를 정의하세요.
LangGraph는 노드가 이 필드를 반환하면 기존 값을 새 값으로 교체합니다.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: _replace(existing, new) 함수는 new를 그대로 반환합니다
# 힌트 2: LangGraph가 state 업데이트 시 이 함수를 호출합니다

def _replace(existing, new):
    """기존 값을 새 값으로 교체하는 reducer."""
    # TODO: 구현 (한 줄이면 충분합니다)
    pass

In [ ]:
# 검증 셀
assert _replace("old", "new") == "new", "새 값을 반환해야 합니다"
assert _replace(None, "value") == "value", "None 기존값도 새 값으로 교체해야 합니다"
assert _replace([1, 2], [3, 4]) == [3, 4], "리스트도 교체해야 합니다"
print("✅ 실습 1 완료! _replace reducer가 올바릅니다.")

## 실습 2: Annotated Reducer가 적용된 State 정의

다음 필드를 가진 `ChatState` TypedDict를 정의하세요:
- `query: Annotated[str, _replace]` - 사용자 질문
- `messages: Annotated[list[str], operator.add]` - 메시지 누적 리스트
- `current_step: Annotated[str, _replace]` - 현재 단계

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: TypedDict를 상속하여 ChatState 클래스 정의
# 힌트 2: Annotated[타입, reducer] 형태로 필드 정의
# 힌트 3: messages는 operator.add를 reducer로 사용 (리스트 += 리스트)

class ChatState(TypedDict):
    pass  # TODO: 3개 필드 정의

## 실습 3: Annotated Reducer를 사용하는 그래프 구현

3개의 노드(greet, ask, respond)를 가진 간단한 챗 그래프를 구현하세요.
각 노드는 `messages` 리스트에 새 메시지를 추가합니다.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: 각 노드는 변경된 필드만 반환합니다 ({**state} 불필요!)
# 힌트 2: messages 필드에 새 메시지 리스트를 반환하면 operator.add로 누적됨
# 힌트 3: StateGraph(ChatState)로 그래프 생성

def greet_node(state: ChatState) -> dict:
    """인사 메시지를 추가합니다."""
    # TODO: {"messages": ["안녕하세요!"], "current_step": "greet"} 반환
    pass

def ask_node(state: ChatState) -> dict:
    """질문 메시지를 추가합니다."""
    # TODO: {"messages": ["무엇을 도와드릴까요?"], "current_step": "ask"} 반환
    pass

def respond_node(state: ChatState) -> dict:
    """응답 메시지를 추가합니다."""
    # TODO: query를 참조하여 응답 생성
    pass

# 그래프 구성
# TODO: graph = StateGraph(ChatState)
# TODO: 노드 추가, 엣지 연결
# TODO: app = graph.compile()

In [ ]:
# 검증 셀
initial_state = {
    "query": "Python에 대해 알려주세요",
    "messages": [],
    "current_step": "start",
}

result = app.invoke(initial_state)

assert "messages" in result, "messages 필드가 없습니다"
assert len(result["messages"]) >= 3, f"최소 3개 메시지가 있어야 합니다. 현재: {len(result['messages'])}"
assert result["current_step"] == "respond", f"마지막 단계가 'respond'여야 합니다. 현재: {result['current_step']}"

print("메시지 목록:")
for i, msg in enumerate(result["messages"]):
    print(f"  {i+1}. {msg}")
print("✅ 실습 3 완료! Annotated Reducer 패턴이 올바르게 작동합니다.")

## 실습 4: operator.add vs _replace 비교

두 reducer의 동작 차이를 확인하세요.

In [ ]:
# operator.add 동작 확인
existing_list = ["메시지1", "메시지2"]
new_list = ["메시지3"]
add_result = operator.add(existing_list, new_list)
replace_result = _replace(existing_list, new_list)

print(f"operator.add: {existing_list} + {new_list} = {add_result}")
print(f"_replace:     {existing_list} -> {new_list} = {replace_result}")

In [ ]:
# 검증 셀
assert add_result == ["메시지1", "메시지2", "메시지3"], "operator.add는 리스트를 합쳐야 합니다"
assert replace_result == ["메시지3"], "_replace는 새 리스트로 교체해야 합니다"
print("✅ 실습 4 완료! operator.add vs _replace 차이를 이해했습니다.")

## 정리

이번 실습에서 배운 내용:
1. `Annotated[타입, reducer]`로 State 필드에 reducer를 지정할 수 있다
2. `operator.add`는 리스트를 누적 추가하고, `_replace`는 새 값으로 교체한다
3. 노드가 변경된 필드만 반환하면 LangGraph가 reducer로 State를 업데이트한다

## 다음 모듈
- **실습 3**: `03_code_compression.ipynb` - AST 기반 코드 압축